## Cuando se pone difícil

En muchas situaciones los archivos pueden tener datos estructurados de alguna manera que no permiten directamente su lectura con `pandas`. 

In [1]:
from pathlib import Path
import pandas as pd
from io import StringIO 

In [2]:
smn_path = Path.cwd() / 'data' / 'smn'
pronostico_file= 'pronostico_5dias20260612.txt'
pronostico_path = smn_path / pronostico_file

Con lo anterior, podemos leer el archivo y crear un diccionario {Ciudad,Dataframe} para que podamos acceder a los datos.

In [3]:
# leer de nuevo el archivo
with open(smn_path / 'pronostico_5dias20260612.txt', 'r') as file:
    content = file.read()

In [5]:

# Dividir el contenido en secciones por aeropuerto
sep =  "================================================================================================"
sections = content.split(sep)
print(f"Secciones encontradas: {len(sections)}")
for i, section in enumerate(sections[0:3]):  # Imprime solo las primeras 3 secciones para no saturar la salida
    print(f"\n--- Sección {i+1} ---")
    print(section[:500])  # Imprime los primeros 500 caracteres de cada sección

Secciones encontradas: 355

--- Sección 1 ---
************************************************************************************************
 Producto basado en un modelo de pronóstico numérico del tiempo, 
 por lo tanto puede diferir del pronostico emitido por el SMN
 ************************************************************************************************
 
 AEROPARQUE
 

--- Sección 2 ---

      FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)                        
 

--- Sección 3 ---

  12/JUN/2026 00Hs.         8.5       347 |   4         0.0 
  12/JUN/2026 03Hs.         7.7       336 |   4         0.0 
  12/JUN/2026 06Hs.         6.8       351 |   5         0.0 
  12/JUN/2026 09Hs.         8.0       344 |   5         0.0 
  12/JUN/2026 12Hs.        14.5       336 |   7         0.0 
  12/JUN/2026 15Hs.        16.4       342 |   6         0.0 
  12/JUN/2026 18Hs.        11.1       348 |   4         0.0 
  

In [6]:
def procesar_seccion_estacion(data_section,columns):
    """Procesa una sección de estación y devuelve un DataFrame."""
    with StringIO(data_section) as data_io:
        df_station = pd.read_fwf(data_io, names=columns)

        df_station = df_station.drop(columns=['l'])

    return df_station


def procesar_pronostico_smn(sections):
    """Crea un diccionario {estacion: DataFrame} a partir de las secciones del archivo."""
    stations_dict = {}
    columns = ['dates', 'h', 't', 'v_dir', 'l', 'v_vel', 'precip']

    # Las secciones siguen el patrón: estacion, header, datos.
    for i in range(0, len(sections), 3):
        if i == 0:
            station_name = sections[i].split('\n')[-2].strip()  
        else:
            station_name = sections[i].strip()
                    

        if i + 2 >= len(sections):
            continue

        data_section = sections[i + 2]

        df_station = procesar_seccion_estacion(data_section, columns)
        stations_dict[station_name] = df_station

    return stations_dict


stations_dict = procesar_pronostico_smn(sections)

print(f"Estaciones procesadas: {list(stations_dict.keys())}")
print(f"\nPrimeras filas de {list(stations_dict.keys())[0]}:")
print(stations_dict[list(stations_dict.keys())[0]].head())


Estaciones procesadas: ['AEROPARQUE', 'AZUL_AERO', 'BAHIA_BLANCA_AERO', 'BARILOCHE_AERO', 'BENITO_JUAREZ_AERO', 'BERNARDO_DE_IRIGOYEN_AERO', 'BOLIVAR_AERO', 'BUENOS_AIRES', 'CATAMARCA_AERO', 'CERES_AERO', 'CHAMICAL_AERO', 'CHAPELCO_AERO', 'CHEPES', 'CHILECITO_AERO', 'CIPOLLETTI', 'COMODORO_RIVADAVIA_AERO', 'CONCORDIA_AERO', 'CORDOBA_AERO', 'CORDOBA_OBSERVATORIO', 'CORONEL_PRINGLES_AERO', 'CORONEL_SUAREZ_AERO', 'CORRIENTES_AERO', 'DOLORES_AERO', 'EL_BOLSON_AERO', 'EL_CALAFATE_AERO', 'EL_PALOMAR_AERO', 'EL_TREBOL', 'ESC.AVIACION_MILITAR_AERO', 'ESQUEL_AERO', 'EZEIZA_AERO', 'FORMOSA_AERO', 'GENERAL_PICO_AERO', 'GOBERNADOR_GREGORES_AERO', 'GUALEGUAYCHU_AERO', 'IGUAZU_AERO', 'ITUZAINGO', 'JACHAL', 'JUJUY_AERO', 'JUJUY_U_N', 'JUNIN_AERO', 'LABOULAYE_AERO', 'LAS_FLORES_AERO', 'LAS_LOMITAS', 'LA_PLATA_AERO', 'LA_QUIACA_OBS', 'LA_RIOJA_AERO', 'MALARGUE_AERO', 'MAQUINCHAO', 'MARCOS_JUAREZ_AERO', 'MARIANO_MORENO_AERO', 'MAR_DEL_PLATA_AERO', 'MENDOZA_AERO', 'MENDOZA_OBSERVATORIO', 'MERCEDES_AERO_(

In [9]:
station = 'TINOGASTA'

df_station = stations_dict[station]
print(df_station)


          dates      h     t  v_dir  v_vel  precip
0   12/JUN/2026  00Hs.   9.5     23      5     0.0
1   12/JUN/2026  03Hs.   8.7     35      8     0.0
2   12/JUN/2026  06Hs.   6.9    334      1     0.0
3   12/JUN/2026  09Hs.   8.0     11      3     0.0
4   12/JUN/2026  12Hs.  13.3     68      5     0.0
5   12/JUN/2026  15Hs.  16.8     68     11     0.0
6   12/JUN/2026  18Hs.  15.2     58     11     0.0
7   12/JUN/2026  21Hs.  12.0     13      6     0.0
8   13/JUN/2026  00Hs.  10.6     86      2     0.0
9   13/JUN/2026  03Hs.   9.6    260      2     0.0
10  13/JUN/2026  06Hs.   8.4     54      2     0.0
11  13/JUN/2026  09Hs.   9.5    293      1     0.0
12  13/JUN/2026  12Hs.  15.7    100      6     0.0
13  13/JUN/2026  15Hs.  18.1     72     11     0.0
14  13/JUN/2026  18Hs.  16.3     51     12     0.0
15  13/JUN/2026  21Hs.  11.2    324      9     0.0
16  14/JUN/2026  00Hs.   9.2    303      6     0.0
17  14/JUN/2026  03Hs.   8.0    267      8     0.0
18  14/JUN/2026  06Hs.   7.7   

In [10]:
# Dame la temperatura a las 0 horas del día 2026-06-16
dia = '15/JUN/2026'
hora = '21Hs.'
df_station[(df_station['dates'] == dia) & (df_station['h'] == hora) ]['t']

31    10.7
Name: t, dtype: float64